## Full Pipeline Notebook

This notebook covers the complete pipeline from raw order-book data to model evaluation:

| Step | Description |
|------|-------------|
| 1 | **Data Splitting** — train/test split from raw CSVs |
| 2 | **Preprocessing** — 100s bucket aggregation, feature extraction |
| 3 | **Noise Floor Analysis** — microstructure noise estimation per stock |
| 4 | **LightGBM Model** — feature engineering, training, evaluation |
| 5 | **Stock Interaction Tests** — statistical tests for cross-stock signals |

> ⚠️ **Note:** The 120s target volatility is extremely small (close to the noise floor), making it difficult to predict reliably. As a result, model performance is degraded, since much of the variation is indistinguishable from microstructure noise.

**Data flow:**
**Data flow:**
```
Raw CSVs → split.py → processed/
         → preprocess.py → processed_v2/
         → LightGBM.py  → models/
         → stock_interaction_test.py → models/
```


## Test Metrics

| Metric        | Value     |
|---------------|----------:|
| MSE (log-RV)  | 0.055747  |
| QLIKE         | 0.028390  |
| RMSE          | 0.000899  |
| MAPE (%)      | 18.135769 |
| RMSPE (%)     | 27.435178 |

---
## 1. Data Splitting

Reads the raw per-stock CSV files and splits all `time_id`s into **80% train / 20% test** using a seeded random shuffle.

All 112 stocks share the same `time_id` universe, so the split is done once on `time_ids` from the first file and applied to all stocks — ensuring no time_id leaks between splits.

**Output:** `processed/train.parquet`, `processed/test.parquet`


### 1.1 Imports and Configuration

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

TEST_RATIO = 0.2
SEED       = 42

### 1.2 `split_and_save` — Train/Test Split Function

In [ ]:
def split_and_save(data_dir: str,
                   output_dir: str,
                   test_ratio: float = TEST_RATIO,
                   seed: int = SEED):

    files = sorted(glob.glob(os.path.join(data_dir, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files in {data_dir}")

    # Get time_ids from first file (shared across all stocks)
    tids = pd.read_csv(files[0], usecols=["time_id"])["time_id"].unique()
    rng  = np.random.default_rng(seed)
    rng.shuffle(tids)

    n_test    = int(len(tids) * test_ratio)
    test_ids  = set(tids[:n_test])
    train_ids = set(tids[n_test:])
    assert train_ids.isdisjoint(test_ids)

    print(f"{len(tids)} time_ids → {len(train_ids)} train, {len(test_ids)} test")

    os.makedirs(output_dir, exist_ok=True)

    train_path = os.path.join(output_dir, "train.parquet")
    test_path  = os.path.join(output_dir, "test.parquet")

    train_writer = None
    test_writer  = None

    for i, f in enumerate(files):
        stock_id = os.path.basename(f).replace(".csv", "")
        df = pd.read_csv(f)
        df["stock_id"] = stock_id

        train_chunk = df[df["time_id"].isin(train_ids)].reset_index(drop=True)
        test_chunk  = df[df["time_id"].isin(test_ids)].reset_index(drop=True)

        # Write train chunk
        table = pa.Table.from_pandas(train_chunk, preserve_index=False)
        if train_writer is None:
            train_writer = pq.ParquetWriter(train_path, table.schema)
        train_writer.write_table(table)

        # Write test chunk
        table = pa.Table.from_pandas(test_chunk, preserve_index=False)
        if test_writer is None:
            test_writer = pq.ParquetWriter(test_path, table.schema)
        test_writer.write_table(table)

        print(f"[{i+1}/{len(files)}] {stock_id} — train {len(train_chunk)}, test {len(test_chunk)}")

        del df, train_chunk, test_chunk, table

    if train_writer:
        train_writer.close()
    if test_writer:
        test_writer.close()

    print(f"\nDone → {train_path}")
    print(f"     → {test_path}")

### 1.3 Run the Split

> **Update `data_dir`** to point to your raw CSV folder before running.


In [ ]:
split_and_save(
    data_dir   = r"C:\Users\DELL\OneDrive - The University of Sydney (Students)\individual_book_train_denorm",
    output_dir = "processed",
    test_ratio = TEST_RATIO,
    seed       = SEED,
)

---
## 2. Preprocessing — Bucket Aggregation

Converts raw per-second order-book data into **3 × 100-second buckets** per stock per `time_id`.

### Why 100-second buckets?
- Per-second granularity or smaller  produces near-zero RV for most illiquid stocks
- 100s buckets require a stock to have zero trades for a **full 100 seconds** to produce RV = 0
- This dramatically reduces noise-dominated samples while preserving temporal structure

### Features per bucket (× 3 buckets = 12 total)
| Feature | Description |
|---------|-------------|
| `log_return` | Net price direction across bucket |
| `realized_vol` | √(Σ r²) — volatility magnitude |
| `mean_spread` | Average bid-ask spread — liquidity proxy |
| `total_volume` | Total traded volume — activity proxy |

### Target
`log(RV)` computed over seconds 300–599 (next 300 seconds)

**Output:** `processed_v2/{train,val,test}_{features,targets}.parquet`


### 2.1 Imports and Configuration

In [ ]:
from __future__ import annotations

import json
import warnings
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ─────────────────────────────────────────────────── Config ──

INPUT_DIR  = Path("processed")           # output of split_and_save.py
OUTPUT_DIR = Path("processed_v2")

BUCKET_SIZE   = 100    # seconds per bucket
N_BUCKETS     = 3      # 3 × 100s = 300s input window
INPUT_SECONDS = BUCKET_SIZE * N_BUCKETS  # 300
TARGET_START  = 300    # seconds 300–599 used for target RV

VAL_RATIO   = 0.1      # fraction of train time_ids held out for validation
RANDOM_SEED = 42
RV_FLOOR    = 1e-8     # floor for TARGET log(RV) only
N_WORKERS   = 8        # parallel workers (tune to your CPU)

# Feature names in order — 4 per bucket, 3 buckets = 12 total
BUCKET_FEATURE_NAMES = ["log_return", "realized_vol", "mean_spread", "total_volume"]

FEATURE_COLS = [
    f"{feat}_b{b}"
    for b in range(N_BUCKETS)
    for feat in BUCKET_FEATURE_NAMES
]   # ['log_return_b0', 'realized_vol_b0', ..., 'total_volume_b2']

INPUT_DIM = len(FEATURE_COLS)   # 12


# ─────────────────────────────────────────── Bucket maths ──

### 2.2 Core Math — Log Returns and RV

In [ ]:
def second_level_log_returns(wap: np.ndarray) -> np.ndarray:
    """Compute log returns from WAP array. Returns empty array if < 2 points."""
    if len(wap) < 2:
        return np.array([], dtype=np.float64)
    with np.errstate(divide="ignore", invalid="ignore"):
        lr = np.log(wap[1:] / wap[:-1])
    return np.where(np.isfinite(lr), lr, 0.0)

In [ ]:
def bucket_features(bucket_df: pd.DataFrame) -> dict[str, float]:
    """
    Extract 4 features from a single 100-second bucket.

    Returns zeros for empty/dead buckets — zero realized_vol is a valid
    signal (the stock was completely flat during this bucket).
    """
    if bucket_df.empty:
        return {f: 0.0 for f in BUCKET_FEATURE_NAMES}

    wap    = bucket_df["wap"].to_numpy(dtype=np.float64)
    spread = bucket_df["bid_ask_spread"].to_numpy(dtype=np.float64)
    vol    = bucket_df["total_volume"].to_numpy(dtype=np.float64)

    lr = second_level_log_returns(wap)

    # 1. log_return — net price move across bucket
    if len(wap) >= 2 and wap[0] > 0 and wap[-1] > 0:
        log_return = float(np.log(wap[-1] / wap[0]))
    else:
        log_return = 0.0

    # 2. realized_vol — sqrt(Σ r²) — zero is valid, do NOT apply floor here
    realized_vol = float(np.sqrt(np.sum(lr ** 2))) if len(lr) > 0 else 0.0

    # 3. mean_spread — average bid-ask spread
    valid_spread = spread[np.isfinite(spread) & (spread > 0)]
    mean_spread  = float(np.mean(valid_spread)) if len(valid_spread) > 0 else 0.0

    # 4. total_volume — total traded volume in bucket
    total_volume = float(np.nansum(vol))

    return {
        "log_return"  : log_return,
        "realized_vol": realized_vol,
        "mean_spread" : mean_spread,
        "total_volume": total_volume,
    }

In [ ]:
def extract_bucketed_features(input_df: pd.DataFrame) -> dict[str, float]:
    """
    Split the 300s input window into N_BUCKETS x BUCKET_SIZE second buckets.
    Returns a flat dict of {feat_bN: value} for all buckets and features.
    """
    result: dict[str, float] = {}

    for b in range(N_BUCKETS):
        start = b * BUCKET_SIZE       # 0, 100, 200
        end   = start + BUCKET_SIZE   # 100, 200, 300

        bucket_df = input_df[
            (input_df["seconds_in_bucket"] >= start) &
            (input_df["seconds_in_bucket"] <  end)
        ]

        feats = bucket_features(bucket_df)
        for feat_name, val in feats.items():
            result[f"{feat_name}_b{b}"] = val

    return result

In [ ]:
def compute_target_log_rv(target_df: pd.DataFrame) -> float | None:
    """
    Compute log(RV) over the target window (seconds 300–599).
    Returns None if no target data is available.
    """
    if target_df.empty:
        return None

    wap = target_df["wap"].to_numpy(dtype=np.float64)
    lr  = second_level_log_returns(wap)
    rv  = float(np.sqrt(np.sum(lr ** 2))) if len(lr) > 0 else 0.0

    # Apply RV_FLOOR only to target — prevents log(0) = -inf
    return float(np.log(max(rv, RV_FLOOR)))


# ─────────────────────────── Per-stock worker function ──

### 2.3 Per-Stock Worker and Parallel Processing

In [ ]:
def process_stock(args: tuple) -> pd.DataFrame | None:
    """
    Worker function — processes all time_ids for a single stock.

    Args:
        args: (stock_id, stock_df) where stock_df is the full data for that stock

    Returns:
        DataFrame with columns: time_id, stock_id, *FEATURE_COLS, log_rv_target
        or None if no valid rows.
    """
    stock_id, stock_df = args

    stock_df = stock_df.sort_values("seconds_in_bucket").reset_index(drop=True)

    rows    = []
    skipped = 0

    for tid, tid_df in stock_df.groupby("time_id"):
        tid_df = tid_df.sort_values("seconds_in_bucket").reset_index(drop=True)

        # Split into input (0–299s) and target (300–599s)
        input_df  = tid_df[tid_df["seconds_in_bucket"] <  INPUT_SECONDS]
        target_df = tid_df[tid_df["seconds_in_bucket"] >= TARGET_START]

        # Skip if no input data at all
        if input_df.empty:
            skipped += 1
            continue

        # Skip if no target data (can't train without a label)
        log_rv_target = compute_target_log_rv(target_df)
        if log_rv_target is None:
            skipped += 1
            continue

        feats = extract_bucketed_features(input_df)

        row = {
            "time_id"      : int(tid),
            "stock_id"     : int(stock_id),
            **feats,
            "log_rv_target": log_rv_target,
        }
        rows.append(row)

    return pd.DataFrame(rows) if rows else None


# ──────────────────────────────────────── Main pipeline ──

In [ ]:
def process_split(raw_df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    """
    Process one split (train/val/test) through bucketing in parallel.
    Groups by stock_id, dispatches each stock to a worker, collects results.
    """
    n_stocks  = raw_df["stock_id"].nunique()
    n_timeids = raw_df["time_id"].nunique()
    print(f"[{split_name}] Processing {n_stocks} stocks x {n_timeids} time_ids ...")

    stock_groups = [
        (sid, grp.copy())
        for sid, grp in raw_df.groupby("stock_id")
    ]

    with Pool(processes=N_WORKERS) as pool:
        results = pool.map(process_stock, stock_groups)

    valid = [r for r in results if r is not None and not r.empty]
    if not valid:
        raise RuntimeError(f"[{split_name}] No data processed successfully.")

    combined = pd.concat(valid, ignore_index=True)
    print(
        f"[{split_name}] Done: {len(combined):,} rows | "
        f"{combined['stock_id'].nunique()} stocks | "
        f"{combined['time_id'].nunique()} time_ids"
    )
    return combined

In [ ]:
def save_split(df: pd.DataFrame, split_name: str) -> None:
    """Save features and targets as separate parquet files."""
    id_cols = ["time_id", "stock_id"]

    feat_path = OUTPUT_DIR / f"{split_name}_features.parquet"
    targ_path = OUTPUT_DIR / f"{split_name}_targets.parquet"

    df[id_cols + FEATURE_COLS].to_parquet(
        feat_path, engine="pyarrow", index=False, compression="snappy"
    )
    df[id_cols + ["log_rv_target"]].to_parquet(
        targ_path, engine="pyarrow", index=False, compression="snappy"
    )

    print(f"  -> {feat_path.name}  ({len(df):,} rows, {len(FEATURE_COLS)} feature cols)")
    print(f"  -> {targ_path.name}")

In [ ]:
def print_target_stats(df: pd.DataFrame, split_name: str) -> None:
    """Print target RV distribution stats and warn if too many floor values."""
    tgt = df["log_rv_target"]
    print(
        f"[{split_name}] log_rv_target stats: "
        f"min={tgt.min():.3f}  p5={tgt.quantile(0.05):.3f}  "
        f"median={tgt.median():.3f}  p95={tgt.quantile(0.95):.3f}  "
        f"max={tgt.max():.3f}"
    )
    floor_pct = (tgt <= np.log(RV_FLOOR) + 0.01).mean() * 100
    if floor_pct > 5:
        print(
            f"  WARNING [{split_name}]: {floor_pct:.1f}% of targets are at RV_FLOOR — "
            f"consider raising BUCKET_SIZE or RV_FLOOR"
        )

### 2.4 Run Preprocessing

In [ ]:
print("=" * 60)
print("Preprocess v2 — 100s bucket aggregation")
print(f"  Buckets     : {N_BUCKETS} x {BUCKET_SIZE}s = {INPUT_SECONDS}s input window")
print(f"  Target      : seconds {TARGET_START}-599 -> log(RV)")
print(f"  Features    : {BUCKET_FEATURE_NAMES} x {N_BUCKETS} buckets = {INPUT_DIM} dims")
print(f"  Workers     : {N_WORKERS}")
print(f"  Input dir   : {INPUT_DIR}")
print(f"  Output dir  : {OUTPUT_DIR}")
print("=" * 60)

# ── Step 1: Load raw parquet splits ────────────────────────────
train_path = INPUT_DIR / "train.parquet"
test_path  = INPUT_DIR / "test.parquet"

if not train_path.exists():
    raise FileNotFoundError(f"Missing: {train_path}")
if not test_path.exists():
    raise FileNotFoundError(f"Missing: {test_path}")

print("Loading parquet files ...")
train_raw = pd.read_parquet(train_path, engine="fastparquet")
test_raw  = pd.read_parquet(test_path,  engine="fastparquet")

# Ensure stock_id is integer.
# split_and_save.py may store it as 'stock_0', 'stock_1', ... or plain int.
def parse_stock_id(s):
    s = str(s)
    return int(s.replace("stock_", "")) if "stock_" in s else int(s)

train_raw["stock_id"] = train_raw["stock_id"].apply(parse_stock_id)
test_raw["stock_id"]  = test_raw["stock_id"].apply(parse_stock_id)

print(
    f"Train raw : {len(train_raw):,} rows | "
    f"{train_raw['stock_id'].nunique()} stocks | "
    f"{train_raw['time_id'].nunique()} time_ids"
)
print(
    f"Test  raw : {len(test_raw):,} rows | "
    f"{test_raw['stock_id'].nunique()} stocks | "
    f"{test_raw['time_id'].nunique()} time_ids"
)

# ── Step 2: Carve validation set from train time_ids ───────────
all_train_tids = train_raw["time_id"].unique()
rng            = np.random.default_rng(RANDOM_SEED)
shuffled_tids  = rng.permutation(all_train_tids)

n_val      = max(1, int(len(shuffled_tids) * VAL_RATIO))
val_tids   = set(shuffled_tids[:n_val].tolist())
train_tids = set(shuffled_tids[n_val:].tolist())

print(f"Train/val split: {len(train_tids)} train | {len(val_tids)} val time_ids")

train_sub = train_raw[train_raw["time_id"].isin(train_tids)].copy()
val_sub   = train_raw[train_raw["time_id"].isin(val_tids)].copy()

del train_raw   # free memory

# ── Step 3: Process each split ─────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, raw_df in [
    ("train", train_sub),
    ("val",   val_sub),
    ("test",  test_raw),
]:
    processed = process_split(raw_df, split_name)
    print_target_stats(processed, split_name)
    save_split(processed, split_name)

# ── Step 4: Save metadata ───────────────────────────────────────
meta = {
    "bucket_size_s" : BUCKET_SIZE,
    "n_buckets"     : N_BUCKETS,
    "input_seconds" : INPUT_SECONDS,
    "target_start_s": TARGET_START,
    "input_dim"     : INPUT_DIM,
    "feature_cols"  : FEATURE_COLS,
    "rv_floor"      : RV_FLOOR,
    "val_ratio"     : VAL_RATIO,
    "random_seed"   : RANDOM_SEED,
}
meta_path = OUTPUT_DIR / "meta.json"
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)
print(f"Metadata saved -> {meta_path}")
print("All splits processed. Done.")

---
## 3. Noise Floor Analysis

Estimates the **minimum meaningful realized volatility** for each stock based on market microstructure noise theory.

### Theory (Kurisu 2018, Roll 1984)
Observed WAP = True mid-price + bid-ask bounce noise

```
Var(noise) ≈ (spread/2)²  per second
RV_noise_floor = √(2 × Var(noise) × window_seconds)
```

Any RV below the noise floor is measuring **bid-ask bounce, not true price movement**.

### Signal-to-Noise Ratio (SNR)
| SNR | Quality | Interpretation |
|-----|---------|----------------|
| > 10 | HIGH | Reliable prediction |
| 2–10 | MARGINAL | Use with caution |
| < 2 | NOISE | Prediction unreliable |

**Output:** `models/stock_noise_profiles.csv`, `models/sample_snr.csv`


### 3.1 Imports and Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

FEATURES_DIR = Path("processed_v2")
OUTPUT_DIR   = Path("models")

# ── Config ──────────────────────────────────────────────────────────
BUCKET_SIZE    = 100    # seconds per bucket
N_BUCKETS      = 3      # number of input buckets
WINDOW_SECONDS = BUCKET_SIZE * N_BUCKETS   # 300s input window

# SNR thresholds from Li et al. (2018) empirical findings
NSR_LIQUID     = 1e-3   # Citigroup-like liquid stocks
NSR_ILLIQUID   = 1e-2   # less liquid stocks

# Quality tiers
SNR_HIGH_THRESHOLD   = 10.0   # RV is 10x noise floor — reliable
SNR_MARGINAL_THRESHOLD = 2.0  # RV is 2x noise floor — use with caution


# ── Spread-based noise estimator (Kurisu 2018) ───────────────────────

### 3.2 Noise Estimation Functions

In [ ]:
def estimate_noise_spread(mean_spread: float, window_seconds: int = WINDOW_SECONDS) -> dict:
    """
    Estimate noise variance from bid-ask spread.

    Theory: observed WAP = true mid-price + bid-ask bounce
    Noise std per trade ~ half the spread (Roll 1984 model)
    So Var(U) per second ~ (spread/2)^2

    RV_noise_floor = sqrt(2 * Var(U) * T)
    where T = number of seconds in the window (from Kurisu eq 4).
    """
    noise_std    = mean_spread / 2.0
    zeta_hat     = noise_std ** 2          # Var(U) per second
    rv_floor     = np.sqrt(2 * zeta_hat * window_seconds)
    log_rv_floor = np.log(max(rv_floor, 1e-12))

    return {
        "zeta_spread" : zeta_hat,
        "rv_floor_spread" : rv_floor,
        "log_rv_floor_spread": log_rv_floor,
    }


# ── RV-based noise estimator (Li et al. 2018, Proposition 3.1) ───────

In [ ]:
def estimate_noise_rv(log_returns: np.ndarray, jn: int = 5) -> dict:
    """
    Estimate Var(U) from lagged realized volatility (eq 7-8 of Li et al.).

    h[Y,Y]^(jn) = sum((Y_{i+jn} - Y_i)^2) / (2*(n - jn + 1))
               P→  Var(U) - gamma(jn)
               ≈  Var(U)  when jn is large enough that gamma(jn) ≈ 0

    At 1-second resolution, noise is ~i.i.d. (Li et al. Section 7.5.1),
    so gamma(jn) ≈ 0 for jn >= 3.  We use jn=5 as default.

    Args:
        log_returns: array of per-second log returns (length T)
        jn:         lag for the RV estimator (default 5 seconds)
    """
    n = len(log_returns)
    if n < jn + 2:
        return {"zeta_rv": np.nan, "rv_floor_rv": np.nan, "log_rv_floor_rv": np.nan}

    # Lagged squared differences: (Y_{i+jn} - Y_i)^2 = (sum of j returns)^2
    # Approximate via sum of returns over jn steps
    lagged_sums = np.array([
        log_returns[i:i+jn].sum()
        for i in range(n - jn)
    ])
    hYY = np.sum(lagged_sums ** 2) / (2 * (n - jn + 1))

    # hYY ≈ Var(U) when noise is i.i.d. and jn is large enough
    zeta_hat     = max(hYY, 0.0)
    rv_floor     = np.sqrt(2 * zeta_hat * WINDOW_SECONDS)
    log_rv_floor = np.log(max(rv_floor, 1e-12))

    return {
        "zeta_rv"        : zeta_hat,
        "rv_floor_rv"    : rv_floor,
        "log_rv_floor_rv": log_rv_floor,
    }


# ── Quality classification ───────────────────────────────────────────

In [ ]:
def classify_quality(rv: float, rv_floor: float) -> str:
    """
    Classify a prediction's reliability based on signal-to-noise ratio.

    SNR = RV / RV_noise_floor
        > 10   : HIGH     — prediction is reliable
        2 to 10: MARGINAL — use with caution, widen confidence intervals
        < 2    : NOISE    — RV is noise-dominated, prediction unreliable
    """
    if rv_floor <= 0 or rv <= 0:
        return "UNKNOWN"
    snr = rv / rv_floor
    if snr >= SNR_HIGH_THRESHOLD:
        return "HIGH"
    elif snr >= SNR_MARGINAL_THRESHOLD:
        return "MARGINAL"
    else:
        return "NOISE"


# ── Per-stock aggregate noise estimates ──────────────────────────────

In [ ]:
def compute_stock_noise_profiles(features_df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute per-stock noise profile by averaging across all time_ids.

    Returns one row per stock with:
        mean_spread   — average bid-ask spread
        zeta_spread   — noise variance estimate (spread method)
        rv_floor      — minimum meaningful RV
        log_rv_floor  — log(rv_floor) — use as LOG_RV_FLOOR in model
    """
    spread_cols = [f"mean_spread_b{b}" for b in range(N_BUCKETS)]
    spread_cols = [c for c in spread_cols if c in features_df.columns]

    if not spread_cols:
        print("WARNING: no mean_spread columns found in features. Using global default.")
        return pd.DataFrame()

    rows = []
    for stock_id, grp in features_df.groupby("stock_id"):
        mean_spread = grp[spread_cols].values.mean()
        est         = estimate_noise_spread(mean_spread)
        rows.append({
            "stock_id"    : stock_id,
            "mean_spread" : mean_spread,
            **est,
        })

    return pd.DataFrame(rows).sort_values("stock_id").reset_index(drop=True)


# ── Per (time_id, stock_id) SNR ──────────────────────────────────────

In [ ]:
def compute_snr_per_sample(
    features_df : pd.DataFrame,
    targets_df  : pd.DataFrame,
    stock_profiles: pd.DataFrame,
) -> pd.DataFrame:
    """
    Compute SNR for every (time_id, stock_id) pair.

    Uses per-stock noise floor from stock_profiles and the actual
    target RV to determine if each sample is signal or noise dominated.
    """
    df = targets_df.merge(
        features_df[["time_id", "stock_id"] +
                    [f"mean_spread_b{b}" for b in range(N_BUCKETS)
                     if f"mean_spread_b{b}" in features_df.columns]],
        on=["time_id", "stock_id"], how="left"
    )

    # Use per-time_id spread if available, else fall back to stock profile
    spread_cols = [f"mean_spread_b{b}" for b in range(N_BUCKETS)
                   if f"mean_spread_b{b}" in df.columns]
    if spread_cols:
        df["mean_spread_sample"] = df[spread_cols].mean(axis=1)
    else:
        df = df.merge(
            stock_profiles[["stock_id", "mean_spread"]],
            on="stock_id", how="left"
        )
        df["mean_spread_sample"] = df["mean_spread"]

    # Noise floor per sample
    df["zeta_sample"]     = (df["mean_spread_sample"] / 2) ** 2
    df["rv_floor_sample"] = np.sqrt(2 * df["zeta_sample"] * WINDOW_SECONDS)

    # Actual RV from target
    df["rv_target"] = np.exp(df["log_rv_target"].clip(-20, 5))

    # SNR and quality
    df["snr"]     = df["rv_target"] / df["rv_floor_sample"].clip(lower=1e-12)
    df["quality"] = df.apply(
        lambda r: classify_quality(r["rv_target"], r["rv_floor_sample"]), axis=1
    )

    return df[["time_id", "stock_id", "log_rv_target", "rv_target",
               "mean_spread_sample", "zeta_sample", "rv_floor_sample",
               "snr", "quality"]]


# ── Main ─────────────────────────────────────────────────────────────

### 3.3 Run Noise Floor Estimation

In [ ]:
print("=" * 60)
print("Noise Floor Estimation")
print(f"  Window : {WINDOW_SECONDS}s ({N_BUCKETS} x {BUCKET_SIZE}s buckets)")
print(f"  Method : spread-based (Roll 1984 + Kurisu 2018)")
print("=" * 60)

print("Loading features and targets ...")
train_feat = pd.read_parquet(FEATURES_DIR / "train_features.parquet")
train_targ = pd.read_parquet(FEATURES_DIR / "train_targets.parquet")

# ── Step 1: Per-stock noise profiles ──────────────────────────
print("\nStep 1: Computing per-stock noise profiles ...")
stock_profiles = compute_stock_noise_profiles(train_feat)

if stock_profiles.empty:
    print("Could not compute stock profiles. Exiting.")
    return

print(f"\nPer-stock noise floor summary:")
print(f"  mean spread         : {stock_profiles['mean_spread'].mean():.6f}")
print(f"  mean zeta (spread)  : {stock_profiles['zeta_spread'].mean():.2e}")
print(f"  mean rv_floor       : {stock_profiles['rv_floor_spread'].mean():.6f}")
print(f"  mean log(rv_floor)  : {stock_profiles['log_rv_floor_spread'].mean():.3f}")
print(f"\n  Suggested LOG_RV_FLOOR for model.py:")
suggested_floor = stock_profiles["log_rv_floor_spread"].quantile(0.75)
print(f"  LOG_RV_FLOOR = {suggested_floor:.2f}  (75th percentile of per-stock floors)")
print(f"  (current hardcoded value: -18.0  --  likely too low)")

# ── Step 2: Per-sample SNR ────────────────────────────────────
print("\nStep 2: Computing per-sample SNR ...")
snr_df = compute_snr_per_sample(train_feat, train_targ, stock_profiles)

# Exclude dead stocks (log_rv_target == 0 or very low)
live = snr_df["log_rv_target"] > -18.0
snr_live = snr_df[live]

print(f"\nSNR distribution (live stocks only, n={len(snr_live):,}):")
print(f"  SNR < 1   (noise dominated) : {(snr_live['snr'] < 1).mean()*100:.1f}%")
print(f"  SNR 1-2   (marginal)        : {((snr_live['snr'] >= 1) & (snr_live['snr'] < 2)).mean()*100:.1f}%")
print(f"  SNR 2-10  (marginal-high)   : {((snr_live['snr'] >= 2) & (snr_live['snr'] < 10)).mean()*100:.1f}%")
print(f"  SNR > 10  (high quality)    : {(snr_live['snr'] >= 10).mean()*100:.1f}%")

print(f"\nQuality breakdown:")
print(snr_live["quality"].value_counts().to_string())

print(f"\nSNR percentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  p{p:2d}: {snr_live['snr'].quantile(p/100):.2f}")

# ── Step 3: What LOG_RV_FLOOR should you use? ─────────────────
print("\nStep 3: Recommended LOG_RV_FLOOR thresholds:")
for pct in [50, 75, 90]:
    floor = stock_profiles["rv_floor_spread"].quantile(pct / 100)
    log_floor = np.log(floor)
    n_excluded = (snr_live["rv_floor_sample"] > snr_live["rv_target"]).mean() * 100
    print(f"  p{pct} noise floor: rv={floor:.2e}  log={log_floor:.2f}")

print(f"\n  Currently you exclude {(snr_live['log_rv_target'] < -18).mean()*100:.1f}% of live rows")
noise_dominated_pct = (snr_live["snr"] < 2).mean() * 100
print(f"  Noise-dominated rows (SNR<2): {noise_dominated_pct:.1f}%")
print(f"  These are samples the model CANNOT learn from reliably.")

# ── Step 4: Save outputs ──────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stock_profiles.to_csv(OUTPUT_DIR / "stock_noise_profiles.csv", index=False)
snr_df.to_csv(OUTPUT_DIR / "sample_snr.csv", index=False)

print(f"\nSaved:")
print(f"  -> {OUTPUT_DIR}/stock_noise_profiles.csv  (per-stock noise floor)")
print(f"  -> {OUTPUT_DIR}/sample_snr.csv            (per-sample SNR + quality flag)")

# ── Step 5: Suggest model config update ──────────────────────
print("\n" + "=" * 60)
print("Recommended model.py config updates:")
rec_floor = stock_profiles["log_rv_floor_spread"].quantile(0.50)
print(f"  LOG_RV_FLOOR = {rec_floor:.1f}   # median per-stock noise floor")
print(f"  (replaces hardcoded -18.0 which is far below any real noise floor)")
print(f"")
print(f"  In RMSPELoss and masked_mse_loss:")
print(f"    live = (target != 0.0) & (target > LOG_RV_FLOOR)")
print(f"")
print(f"  This will exclude {noise_dominated_pct:.0f}% of noise-dominated samples")
print(f"  from the loss, reducing the signal your model trains on but")
print(f"  improving the quality of each gradient update.")
print("=" * 60)

---
## 4. LightGBM Realized Volatility Forecaster

A gradient-boosted tree model trained on engineered per-stock features.

### Why LightGBM?
- Captures non-linear per-stock feature interactions
- Handles categorical `stock_id` directly via native categorical splits
- No cross-stock graph needed — leverages per-stock fixed effects
- Fast training with early stopping

### Feature Groups (37 features total)
| Group | Features | What it captures |
|-------|----------|-----------------|
| HAR log-RV | `log_rv_b*`, `rv_level/trend/accel` | Volatility persistence |
| Spread | `spread_level/trend/accel` | Liquidity regime |
| Volume | `volume_trend`, `log_vol_trend` | Activity regime |
| Returns | `return_sum`, `return_trend` | Price momentum |
| Interactions | `amihud_b*`, `snr_b1`, `spread_adj_rv` | Microstructure |
| Market | `market_rv_mean`, `stock_vs_market` | Cross-sectional regime |
| Identity | `stock_id_cat` | Per-stock fixed effects |

**Output:** `models/lgbm_model.pkl`, `models/lgbm_predictions.csv`


### 4.1 Imports, Configuration and Parameter Loading

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pickle
import lightgbm as lgb

EPS          = 1e-8
LOG_RV_FLOOR = -18.0
RV_FLOOR     = 1e-8

FEATURES_DIR   = Path("processed_v2")
OUTPUT_DIR     = Path("models")
BEST_PARAMS_PATH = OUTPUT_DIR / "lgbm_best_params.json"

N_BUCKETS    = 3
BUCKET_FEATS = ["log_return", "realized_vol", "mean_spread", "total_volume"]

# Default params — used only if lgbm_best_params.json is not found
LGBM_PARAMS_DEFAULT = {
    "objective"        : "regression",
    "metric"           : "rmse",
    "boosting_type"    : "gbdt",
    "num_leaves"       : 63,
    "max_depth"        : -1,
    "learning_rate"    : 0.05,
    "n_estimators"     : 1000,
    "min_child_samples": 50,
    "subsample"        : 0.8,
    "subsample_freq"   : 1,
    "colsample_bytree" : 0.8,
    "reg_alpha"        : 0.1,
    "reg_lambda"       : 1.0,
    "random_state"     : 42,
    "n_jobs"           : -1,
    "verbose"          : -1,
}

### 4.2 Load Tuned Parameters

In [ ]:
def load_params() -> dict:
    """
    Load tuned params from lgbm_best_params.json if available,
    otherwise fall back to LGBM_PARAMS_DEFAULT.
    Always ensures required keys (objective, metric, verbose, n_jobs)
    are present so the model trains correctly regardless of what
    lgbm_tune.py saved.
    """
    if BEST_PARAMS_PATH.exists():
        with open(BEST_PARAMS_PATH) as f:
            params = json.load(f)
        print(f"Loaded tuned params from {BEST_PARAMS_PATH}")

        # Ensure required keys that tune script may not include
        params.setdefault("objective",  "regression")
        params.setdefault("metric",     "rmse")
        params.setdefault("verbose",    -1)
        params.setdefault("n_jobs",     -1)
        params.setdefault("random_state", 42)
        params.setdefault("subsample_freq", 1)

        # n_estimators from tuning is the best_iteration — use it directly
        # (no early stopping needed since we already know the right value)
        print(f"  boosting_type  : {params.get('boosting_type', 'gbdt')}")
        print(f"  num_leaves     : {params.get('num_leaves')}")
        print(f"  learning_rate  : {params.get('learning_rate', 'N/A'):.4f}")
        print(f"  n_estimators   : {params.get('n_estimators')}")
        print(f"  reg_alpha      : {params.get('reg_alpha', 'N/A'):.4f}")
        print(f"  reg_lambda     : {params.get('reg_lambda', 'N/A'):.4f}")
        return params

    print(f"lgbm_best_params.json not found — using default params.")
    print("Run lgbm_tune.py first to get optimised hyperparameters.")
    return LGBM_PARAMS_DEFAULT.copy()


# ── Metrics ──────────────────────────────────────────────────────────

### 4.3 Evaluation Metrics

In [ ]:
def compute_metrics(pred_log: np.ndarray, true_log: np.ndarray) -> dict:
    pred_log  = np.clip(pred_log, -20, 5)
    mse_log   = float(np.mean((pred_log - true_log) ** 2))
    pred_rv   = np.clip(np.exp(pred_log), EPS, None)
    true_rv   = np.clip(np.exp(true_log), EPS, None)
    resid     = pred_rv - true_rv
    rmse      = float(np.sqrt(np.mean(resid ** 2)))
    mape      = float(np.mean(np.abs(resid) / true_rv) * 100)
    rmspe     = float(np.sqrt(np.mean((resid / true_rv) ** 2)) * 100)
    ratio     = true_rv / pred_rv
    qlike     = float(np.mean(ratio - np.log(ratio) - 1))
    return {
        "MSE (log-RV)": mse_log,
        "QLIKE"       : qlike,
        "RMSE"        : rmse,
        "MAPE%"       : mape,
        "RMSPE%"      : rmspe,
    }


# ── Feature engineering ───────────────────────────────────────────────

### 4.4 Feature Engineering

Builds the full feature matrix from the 3-bucket parquet features.
The `market_rv_mean` and `stock_vs_market` features capture contemporaneous
cross-sectional volatility regime — the strongest cross-stock signal identified
by the interaction tests.


In [ ]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build a rich feature matrix from the 3-bucket parquet features.

    All features are computed per (time_id, stock_id) row.
    LightGBM handles non-linearity so we provide both raw values
    and derived interaction terms — the tree will select what matters.
    """
    out = pd.DataFrame()
    out["time_id"]  = df["time_id"]
    out["stock_id"] = df["stock_id"]

    # ── Raw bucket features ───────────────────────────────────────
    for b in range(N_BUCKETS):
        for feat in BUCKET_FEATS:
            col = f"{feat}_b{b}"
            if col in df.columns:
                out[col] = df[col].fillna(0.0)

    # ── Log-scale RV (matches HAR specification) ──────────────────
    for b in range(N_BUCKETS):
        col = f"realized_vol_b{b}"
        if col in df.columns:
            out[f"log_rv_b{b}"] = np.log(df[col].clip(lower=RV_FLOOR))

    # ── HAR volatility features ───────────────────────────────────
    # Short / medium / long RV in log space
    if all(f"log_rv_b{b}" in out for b in range(N_BUCKETS)):
        out["rv_level"]  = out[["log_rv_b0", "log_rv_b1", "log_rv_b2"]].mean(axis=1)
        out["rv_trend"]  = out["log_rv_b2"] - out["log_rv_b0"]   # recent vs old
        out["rv_accel"]  = (out["log_rv_b2"] - out["log_rv_b1"]) \
                         - (out["log_rv_b1"] - out["log_rv_b0"])  # curvature
        out["rv_max"]    = out[["log_rv_b0", "log_rv_b1", "log_rv_b2"]].max(axis=1)
        out["rv_min"]    = out[["log_rv_b0", "log_rv_b1", "log_rv_b2"]].min(axis=1)
        out["rv_range"]  = out["rv_max"] - out["rv_min"]          # intra-window spread

    # ── Spread features ───────────────────────────────────────────
    spread_cols = [f"mean_spread_b{b}" for b in range(N_BUCKETS)
                   if f"mean_spread_b{b}" in df.columns]
    if len(spread_cols) == N_BUCKETS:
        s0, s1, s2 = [df[c].fillna(0.0) for c in spread_cols]
        out["spread_level"]  = s1                    # mid-window spread
        out["spread_trend"]  = s2 - s0               # tightening or widening
        out["spread_accel"]  = (s2 - s1) - (s1 - s0) # curvature
        out["spread_mean"]   = (s0 + s1 + s2) / 3
        out["spread_max"]    = pd.concat([s0, s1, s2], axis=1).max(axis=1)

    # ── Volume features ───────────────────────────────────────────
    vol_cols = [f"total_volume_b{b}" for b in range(N_BUCKETS)
                if f"total_volume_b{b}" in df.columns]
    if len(vol_cols) == N_BUCKETS:
        v0, v1, v2 = [df[c].fillna(0.0) for c in vol_cols]
        out["volume_level"]  = v1
        out["volume_trend"]  = v2 - v0
        out["volume_accel"]  = (v2 - v1) - (v1 - v0)
        out["volume_total"]  = v0 + v1 + v2
        out["volume_max"]    = pd.concat([v0, v1, v2], axis=1).max(axis=1)
        # Log volume (more normally distributed)
        out["log_vol_b0"]    = np.log(v0.clip(lower=EPS))
        out["log_vol_b1"]    = np.log(v1.clip(lower=EPS))
        out["log_vol_b2"]    = np.log(v2.clip(lower=EPS))
        out["log_vol_trend"] = out["log_vol_b2"] - out["log_vol_b0"]

    # ── Return features ───────────────────────────────────────────
    ret_cols = [f"log_return_b{b}" for b in range(N_BUCKETS)
                if f"log_return_b{b}" in df.columns]
    if len(ret_cols) == N_BUCKETS:
        r0, r1, r2 = [df[c].fillna(0.0) for c in ret_cols]
        out["return_sum"]    = r0 + r1 + r2           # net price move
        out["return_trend"]  = r2 - r0                # momentum
        out["return_abs_b0"] = r0.abs()
        out["return_abs_b1"] = r1.abs()
        out["return_abs_b2"] = r2.abs()
        out["return_abs_sum"] = r0.abs() + r1.abs() + r2.abs()
        # Direction consistency (all same sign = trending)
        out["return_sign_b0"] = np.sign(r0)
        out["return_sign_b1"] = np.sign(r1)
        out["return_sign_b2"] = np.sign(r2)
        out["direction_consistent"] = (
            (np.sign(r0) == np.sign(r1)) & (np.sign(r1) == np.sign(r2))
        ).astype(float)

    # ── Cross-feature interactions ────────────────────────────────
    # Amihud illiquidity: |return| / volume (price impact per unit vol)
    for b in range(N_BUCKETS):
        ret_col = f"log_return_b{b}"
        vol_col = f"total_volume_b{b}"
        if ret_col in df.columns and vol_col in df.columns:
            out[f"amihud_b{b}"] = (
                df[ret_col].abs() / (df[vol_col].clip(lower=EPS))
            ).fillna(0.0)

    # Volatility × volume (active volatile period or just noise?)
    if "realized_vol_b1" in df.columns and "total_volume_b1" in df.columns:
        out["vol_times_volume"] = (
            df["realized_vol_b1"] * df["total_volume_b1"]
        ).fillna(0.0)

    # Spread × RV (liquidity-adjusted volatility)
    if "mean_spread_b1" in df.columns and "realized_vol_b1" in df.columns:
        out["spread_adj_rv"] = (
            df["realized_vol_b1"] / df["mean_spread_b1"].clip(lower=EPS)
        ).fillna(0.0)

    # Noise floor estimate: sqrt(2 * zeta * 300)
    # zeta = (spread/2)^2 per second
    if "mean_spread_b1" in df.columns:
        zeta = (df["mean_spread_b1"] / 2) ** 2
        out["noise_floor"]    = np.sqrt(2 * zeta * 300)
        out["log_noise_floor"] = np.log(out["noise_floor"].clip(lower=EPS))

    # SNR: realized_vol / noise_floor
    if "realized_vol_b1" in df.columns and "noise_floor" in out.columns:
        out["snr_b1"] = (
            df["realized_vol_b1"] / out["noise_floor"].clip(lower=EPS)
        ).fillna(0.0)

    # Stock identity — LightGBM uses this as a categorical split feature
    # Captures per-stock fixed effects (always illiquid, always volatile, etc.)
    out["stock_id_cat"] = df["stock_id"].astype("category")

    tid_rv_mean = df.groupby("time_id")["realized_vol_b2"].transform("mean")
    tid_rv_std  = df.groupby("time_id")["realized_vol_b2"].transform("std")

    out["market_rv_mean"] = tid_rv_mean   # market-wide volatility level
    out["market_rv_std"]  = tid_rv_std    # market-wide volatility dispersion
    out["stock_vs_market"] = (
        df["realized_vol_b2"] - tid_rv_mean
    ) / tid_rv_std.clip(lower=1e-8)       # this stock's deviation from market


    return out

### 4.5 Helper Functions

In [ ]:
def get_feature_cols(df: pd.DataFrame) -> list:
    """Return all feature columns (exclude id and target columns)."""
    exclude = {"time_id", "stock_id", "log_rv_target"}
    return [c for c in df.columns if c not in exclude]


# ── Live mask ────────────────────────────────────────────────────────

In [ ]:
def is_live(series: pd.Series) -> pd.Series:
    """True for stocks with meaningful target RV (not dead or floor)."""
    return (series != 0.0) & (series > LOG_RV_FLOOR)


# ── Main ─────────────────────────────────────────────────────────────

### 4.6 Train and Evaluate LightGBM

In [ ]:
print("=" * 60)
print("LightGBM Realized Volatility Forecaster")
print(f"  Data dir : {FEATURES_DIR}")
print(f"  Output   : {OUTPUT_DIR}")
print("=" * 60)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Step 1: Load parquet files ────────────────────────────────
print("\nLoading parquet files ...")
train_feat = pd.read_parquet(FEATURES_DIR / "train_features.parquet")
train_targ = pd.read_parquet(FEATURES_DIR / "train_targets.parquet")
val_feat   = pd.read_parquet(FEATURES_DIR / "val_features.parquet")
val_targ   = pd.read_parquet(FEATURES_DIR / "val_targets.parquet")
test_feat  = pd.read_parquet(FEATURES_DIR / "test_features.parquet")
test_targ  = pd.read_parquet(FEATURES_DIR / "test_targets.parquet")

print(f"  Train: {len(train_feat):,} rows")
print(f"  Val  : {len(val_feat):,} rows")
print(f"  Test : {len(test_feat):,} rows")

# ── Step 2: Merge features + targets ─────────────────────────
def merge(feat, targ):
    return feat.merge(targ[["time_id", "stock_id", "log_rv_target"]],
                      on=["time_id", "stock_id"], how="inner")

train_df = merge(train_feat, train_targ)
val_df   = merge(val_feat,   val_targ)
test_df  = merge(test_feat,  test_targ)

# ── Step 3: Engineer features ─────────────────────────────────
print("\nEngineering features ...")
train_eng = engineer_features(train_df)
val_eng   = engineer_features(val_df)
test_eng  = engineer_features(test_df)

# Add target back
train_eng["log_rv_target"] = train_df["log_rv_target"].values
val_eng["log_rv_target"]   = val_df["log_rv_target"].values
test_eng["log_rv_target"]  = test_df["log_rv_target"].values

# ── Step 4: Filter live stocks ────────────────────────────────
train_live = train_eng[is_live(train_eng["log_rv_target"])].copy()
val_live   = val_eng[is_live(val_eng["log_rv_target"])].copy()
test_live  = test_eng[is_live(test_eng["log_rv_target"])].copy()

print(f"\nLive rows after filtering:")
print(f"  Train: {len(train_live):,} / {len(train_eng):,}")
print(f"  Val  : {len(val_live):,} / {len(val_eng):,}")
print(f"  Test : {len(test_live):,} / {len(test_eng):,}")

# ── Step 5: Prepare matrices ──────────────────────────────────
feat_cols = get_feature_cols(train_live)
cat_cols  = ["stock_id_cat"] if "stock_id_cat" in feat_cols else []

print(f"\nFeature count: {len(feat_cols)}")

X_train = train_live[feat_cols]
y_train = train_live["log_rv_target"].values

X_val   = val_live[feat_cols]
y_val   = val_live["log_rv_target"].values

X_test  = test_live[feat_cols]
y_test  = test_live["log_rv_target"].values

# ── Step 6: Load params and train LightGBM ─────────────────
params = load_params()
using_tuned = BEST_PARAMS_PATH.exists()

print("\nTraining LightGBM ...")
model = lgb.LGBMRegressor(**params)

if using_tuned:
    # Tuned params already have the right n_estimators from best_iteration.
    # No early stopping needed — train for exactly that many trees.
    model.fit(
        X_train, y_train,
        categorical_feature=cat_cols if cat_cols else "auto",
        callbacks=[lgb.log_evaluation(period=50)],
    )
    print(f"\nTrained for {params.get('n_estimators')} iterations (from tuning).")
else:
    # Default params — use early stopping to find best n_estimators
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=True),
            lgb.log_evaluation(period=50),
        ],
        categorical_feature=cat_cols if cat_cols else "auto",
    )
    print(f"\nBest iteration: {model.best_iteration_}")

# ── Step 7: Evaluate ──────────────────────────────────────────
for split_name, X, y in [
    ("Train", X_train, y_train),
    ("Val",   X_val,   y_val),
    ("Test",  X_test,  y_test),
]:
    pred    = model.predict(X)
    metrics = compute_metrics(pred, y)
    print(f"\n{split_name} metrics:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.6f}")

# ── Step 8: Bias / variance check ────────────────────────────
pred_test = model.predict(X_test)
pred_rv   = np.exp(np.clip(pred_test, -20, 5))
true_rv   = np.exp(y_test)

print(f"\nBias / variance check (test set):")
print(f"  Mean pred RV : {pred_rv.mean():.6f}")
print(f"  Mean true RV : {true_rv.mean():.6f}")
print(f"  Ratio        : {(pred_rv / true_rv.clip(EPS)).mean():.3f}")
print(f"  Std pred RV  : {pred_rv.std():.6f}")
print(f"  Std true RV  : {true_rv.std():.6f}")
print(f"  Var ratio    : {pred_rv.std() / true_rv.std():.3f}")

# ── Step 9: Per-RV-tier metrics ───────────────────────────────
print(f"\nMetrics by RV tier:")
for name, mask in [
    ("Low  (<0.005) ", true_rv < 0.005),
    ("High (>=0.005)", true_rv >= 0.005),
]:
    if mask.sum() == 0:
        continue
    m = compute_metrics(pred_test[mask], y_test[mask])
    print(f"  {name} n={mask.sum():6,}: "
          f"QLIKE={m['QLIKE']:.4f}  MAPE%={m['MAPE%']:.1f}")

# ── Step 10: Feature importance ───────────────────────────────
importance = pd.DataFrame({
    "feature"   : feat_cols,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)

print(f"\nTop 20 features:")
print(importance.head(20).to_string(index=False))

importance.to_csv(OUTPUT_DIR / "lgbm_feature_importance.csv", index=False)

# ── Step 11: Save model ───────────────────────────────────────
model_path = OUTPUT_DIR / "lgbm_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model, f)
print(f"\nModel saved -> {model_path}")

# ── Step 12: Export predictions ───────────────────────────────
out_df = test_live[["time_id", "stock_id"]].copy()
out_df["pred_log_rv"] = np.clip(pred_test, -20, 5)
out_df["true_log_rv"] = y_test
out_df["pred_rv"]     = np.exp(out_df["pred_log_rv"])
out_df["true_rv"]     = np.exp(out_df["true_log_rv"])

out_path = OUTPUT_DIR / "lgbm_predictions.csv"
out_df.to_csv(out_path, index=False)
print(f"Predictions saved -> {out_path} ({len(out_df):,} rows)")

print("\nDone.")

---
## 5. Stock Interaction Tests

**Pure statistical tests — completely independent of any model.**

Tests whether cross-stock signals are genuinely useful for forecasting,
answering a progressively harder question with each test.

| Test | Question | Method |
|------|----------|--------|
| 1 | Do stocks co-move in volatility? | Pearson correlation matrix |
| 2 | Does stock A's past RV predict stock B's future RV? | Granger causality F-test |
| 3 | Is the spatial structure real or random? | Permutation baseline |
| 4 | Is co-movement contemporaneous or lagged? | Lead-lag correlation |

### Results on this dataset
```
Test 1 — Correlation:   0.763 (STRONG)     — stocks highly co-move
Test 2 — Granger:       4.0%  (NOT SIG)    — no predictive lag
Test 3 — Permutation:   z=10820 (REAL)     — spatial structure genuine
Test 4 — Lagged corr:   0.006 (CONTEMP.)   — co-movement is same-time only
```

**Key insight:** Volatility co-movement is entirely contemporaneous.
Cross-stock signals exist but are not forecastable from lagged data.
This explains why LGBM's per-stock `stock_id_cat` beats the GNN's graph.


### 5.1 Imports and Configuration

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
from scipy import stats
from scipy.stats import f as f_dist

EPS          = 1e-8
LOG_RV_FLOOR = -18.0
FEATURES_DIR = Path("processed_v2")
OUTPUT_DIR   = Path("models")

# Granger test config
GRANGER_LAGS    = 1     # how many time_id lags to use
N_STOCK_PAIRS   = 200   # number of random pairs to test (full N² is too slow)
GRANGER_ALPHA   = 0.05  # significance level

# Spillover config
VAR_LAGS        = 1     # VAR lag order
N_STOCKS_VAR    = 20    # subset of stocks for VAR (full 112 is slow)
HORIZON         = 10    # forecast horizon for variance decomposition

# Permutation test
N_PERMUTATIONS  = 100


# ── Load and pivot data ───────────────────────────────────────────────

### 5.2 Data Loading — RV Panel

In [ ]:
def load_rv_panel(split: str = "train") -> pd.DataFrame:
    """
    Load realized volatility targets and pivot to wide format.

    Returns DataFrame with:
      index  : time_id (sorted)
      columns: stock_id
      values : log_rv_target (NaN for missing stocks)
    """
    targ = pd.read_parquet(FEATURES_DIR / f"{split}_targets.parquet")

    # Filter live rows
    live = (targ["log_rv_target"] != 0.0) & (targ["log_rv_target"] > LOG_RV_FLOOR)
    targ = targ[live]

    panel = targ.pivot(index="time_id", columns="stock_id", values="log_rv_target")
    panel = panel.sort_index()

    print(f"Panel shape: {panel.shape[0]} time_ids × {panel.shape[1]} stocks")
    print(f"Sparsity: {panel.isna().mean().mean()*100:.1f}% missing")
    return panel


# ── Test 1: Correlation matrix ────────────────────────────────────────

### 5.3 Test 1 — Cross-Stock RV Correlation

Computes pairwise Pearson correlation of log(RV) across time_ids.
A mean correlation > 0.3 indicates strong market-wide co-movement.


In [ ]:
def test_correlation(panel: pd.DataFrame) -> dict:
    """
    Compute pairwise Pearson correlation of log(RV) across time_ids.
    Stocks that co-move in volatility should show high positive correlation.
    """
    print("\n" + "="*55)
    print("Test 1: Cross-Stock RV Correlation")
    print("="*55)

    # Use stocks with sufficient observations
    obs_count  = panel.notna().sum()
    good_stocks = obs_count[obs_count >= panel.shape[0] * 0.5].index
    sub = panel[good_stocks].dropna(how="all")

    corr_matrix = sub.corr(method="pearson")

    # Extract upper triangle (excluding diagonal)
    n = len(corr_matrix)
    upper = corr_matrix.values[np.triu_indices(n, k=1)]
    upper = upper[~np.isnan(upper)]

    print(f"Stocks analysed: {n}")
    print(f"Stock pairs    : {len(upper):,}")
    print(f"\nPairwise log(RV) correlation:")
    print(f"  Mean        : {upper.mean():.4f}")
    print(f"  Median      : {np.median(upper):.4f}")
    print(f"  Std         : {upper.std():.4f}")
    print(f"  % positive  : {(upper > 0).mean()*100:.1f}%")
    print(f"  % > 0.3     : {(upper > 0.3).mean()*100:.1f}%  (moderate correlation)")
    print(f"  % > 0.5     : {(upper > 0.5).mean()*100:.1f}%  (strong correlation)")

    # Interpretation
    if upper.mean() > 0.3:
        print("\n  FINDING: Strong average cross-stock RV correlation.")
        print("  Stocks move together — cross-stock signals should help forecasting.")
    elif upper.mean() > 0.1:
        print("\n  FINDING: Moderate average cross-stock RV correlation.")
        print("  Some co-movement — cross-stock signals may help on active days.")
    else:
        print("\n  FINDING: Weak average cross-stock RV correlation.")
        print("  Stocks are largely independent — cross-stock signals unlikely to help.")

    # Save correlation matrix
    corr_matrix.to_csv(OUTPUT_DIR / "rv_correlation_matrix.csv")
    print(f"\n  Correlation matrix saved -> models/rv_correlation_matrix.csv")

    return {
        "mean_corr"   : float(upper.mean()),
        "median_corr" : float(np.median(upper)),
        "pct_positive": float((upper > 0).mean()),
        "pct_gt_03"   : float((upper > 0.3).mean()),
        "pct_gt_05"   : float((upper > 0.5).mean()),
        "n_stocks"    : n,
        "n_pairs"     : len(upper),
    }


# ── Test 2: Granger causality ─────────────────────────────────────────

### 5.4 Test 2 — Granger Causality

Tests whether stock A's past RV helps predict stock B's future RV beyond B's own history.

- **Restricted model:** y_t = a + b₁·y_{t-1} + ε
- **Unrestricted model:** y_t = a + b₁·y_{t-1} + c₁·x_{t-1} + ε
- F-test on the difference → low p-value means cross-stock signal is predictive


In [ ]:
def granger_test_pair(
    y: np.ndarray,     # target stock log(RV) series
    x: np.ndarray,     # candidate stock log(RV) series
    lags: int = 1,
) -> float:
    """
    Test whether x Granger-causes y.

    Restricted model:   y_t = a + b1*y_{t-1} + e
    Unrestricted model: y_t = a + b1*y_{t-1} + c1*x_{t-1} + e

    Returns p-value of F-test. Low p-value → x helps predict y.
    """
    # Align and remove NaNs
    df = pd.DataFrame({"y": y, "x": x}).dropna()
    if len(df) < lags + 10:
        return np.nan

    y_arr = df["y"].values
    x_arr = df["x"].values

    n = len(y_arr) - lags

    # Build lagged arrays
    Y     = y_arr[lags:]                    # dependent variable
    Y_lag = y_arr[:-lags].reshape(-1, 1)   # lagged Y
    X_lag = x_arr[:-lags].reshape(-1, 1)   # lagged X
    ones  = np.ones((n, 1))

    # Restricted: Y ~ 1 + Y_lag
    Z_r   = np.hstack([ones, Y_lag])
    beta_r = np.linalg.lstsq(Z_r, Y, rcond=None)[0]
    resid_r = Y - Z_r @ beta_r
    rss_r   = np.sum(resid_r**2)

    # Unrestricted: Y ~ 1 + Y_lag + X_lag
    Z_u   = np.hstack([ones, Y_lag, X_lag])
    beta_u = np.linalg.lstsq(Z_u, Y, rcond=None)[0]
    resid_u = Y - Z_u @ beta_u
    rss_u   = np.sum(resid_u**2)

    # F-statistic
    q  = lags       # number of restrictions
    k  = Z_u.shape[1]
    df1 = q
    df2 = n - k

    if df2 <= 0 or rss_u <= 0:
        return np.nan

    f_stat = ((rss_r - rss_u) / q) / (rss_u / df2)
    p_val  = 1 - f_dist.cdf(f_stat, df1, df2)
    return float(p_val)

In [ ]:
def test_granger(panel: pd.DataFrame) -> dict:
    """
    Test Granger causality on random sample of stock pairs.
    """
    print("\n" + "="*55)
    print("Test 2: Granger Causality")
    print(f"  Lags: {GRANGER_LAGS} | Pairs tested: {N_STOCK_PAIRS} | α={GRANGER_ALPHA}")
    print("="*55)

    stocks = panel.columns.tolist()
    if len(stocks) < 2:
        print("Not enough stocks for Granger test.")
        return {}

    rng = np.random.default_rng(42)
    pairs = [(int(rng.choice(len(stocks))), int(rng.choice(len(stocks))))
             for _ in range(N_STOCK_PAIRS * 3)]
    pairs = [(a, b) for a, b in pairs if a != b][:N_STOCK_PAIRS]

    p_values = []
    for i, (a_idx, b_idx) in enumerate(pairs):
        s_a = stocks[a_idx]
        s_b = stocks[b_idx]
        y = panel[s_b].values   # target: stock B
        x = panel[s_a].values   # predictor: stock A
        p = granger_test_pair(y, x, lags=GRANGER_LAGS)
        if not np.isnan(p):
            p_values.append(p)

    p_values = np.array(p_values)
    pct_significant = float((p_values < GRANGER_ALPHA).mean())

    print(f"Valid pairs tested : {len(p_values)}")
    print(f"Significant pairs  : {(p_values < GRANGER_ALPHA).sum()} "
          f"({pct_significant*100:.1f}%)")
    print(f"Expected by chance : {GRANGER_ALPHA*100:.1f}%")
    print(f"Mean p-value       : {p_values.mean():.4f}")
    print(f"Median p-value     : {np.median(p_values):.4f}")

    # Binomial test: is significant% > alpha% by more than chance?
    binom_result = stats.binomtest(
        int((p_values < GRANGER_ALPHA).sum()),
        len(p_values),
        GRANGER_ALPHA,
        alternative="greater"
    )

    print(f"\nBinomial test p-value: {binom_result.pvalue:.4f}")
    if binom_result.pvalue < 0.05:
        print("  FINDING: Significantly more Granger-causal pairs than chance.")
        print("  Stock interactions are statistically informative for forecasting.")
    else:
        print("  FINDING: No significant excess of Granger-causal pairs.")
        print("  Cross-stock signals may not add reliable forecasting value.")

    return {
        "n_pairs_tested"   : len(p_values),
        "pct_significant"  : pct_significant,
        "expected_by_chance": GRANGER_ALPHA,
        "mean_pvalue"      : float(p_values.mean()),
        "median_pvalue"    : float(np.median(p_values)),
        "binom_pvalue"     : float(binom_result.pvalue),
        "interaction_significant": bool(binom_result.pvalue < 0.05),
    }


# ── Test 3: Permutation baseline ─────────────────────────────────────

### 5.5 Test 3 — Permutation Baseline

Shuffles stock identities within each time_id to destroy spatial structure
while preserving the marginal distribution of RV values.

Real correlation >> null correlation → spatial structure is genuine information.


In [ ]:
def test_permutation(panel: pd.DataFrame) -> dict:
    """
    Shuffle stock labels within each time_id and measure how much
    the correlation structure degrades.

    Real correlation = correlation in observed data
    Null correlation = correlation after shuffling stock identities
                       within each time_id (destroys spatial structure,
                       preserves marginal distribution)

    If real >> null → spatial structure is genuinely informative.
    """
    print("\n" + "="*55)
    print("Test 3: Permutation Baseline")
    print(f"  Shuffles: {N_PERMUTATIONS}")
    print("="*55)

    obs_count   = panel.notna().sum()
    good_stocks = obs_count[obs_count >= panel.shape[0] * 0.5].index
    sub         = panel[good_stocks].dropna(how="all")

    # Real correlation
    corr_real = sub.corr().values
    n         = len(corr_real)
    upper_idx = np.triu_indices(n, k=1)
    real_upper = corr_real[upper_idx]
    real_upper = real_upper[~np.isnan(real_upper)]
    real_mean  = float(real_upper.mean())

    # Permuted correlations
    rng          = np.random.default_rng(42)
    perm_means   = []

    for _ in range(N_PERMUTATIONS):
        shuffled = sub.copy()
        # Shuffle stock columns within each time_id
        for tid in shuffled.index:
            row   = shuffled.loc[tid].values.copy()
            valid = ~np.isnan(row)
            row[valid] = rng.permutation(row[valid])
            shuffled.loc[tid] = row

        corr_perm  = shuffled.corr().values
        perm_upper = corr_perm[upper_idx]
        perm_upper = perm_upper[~np.isnan(perm_upper)]
        perm_means.append(float(perm_upper.mean()))

    perm_means  = np.array(perm_means)
    null_mean   = float(perm_means.mean())
    null_std    = float(perm_means.std())
    z_score     = (real_mean - null_mean) / (null_std + 1e-10)
    p_value     = float(stats.norm.sf(z_score))  # one-sided

    print(f"Real mean correlation     : {real_mean:.4f}")
    print(f"Null mean correlation     : {null_mean:.4f}  (±{null_std:.4f})")
    print(f"Excess correlation        : {real_mean - null_mean:.4f}")
    print(f"Z-score                   : {z_score:.2f}")
    print(f"P-value (one-sided)       : {p_value:.4f}")

    if p_value < 0.05:
        print(f"\n  FINDING: Real correlation ({real_mean:.3f}) significantly exceeds")
        print(f"  null ({null_mean:.3f}). Stock spatial structure is genuine.")
        print(f"  Cross-stock signals carry real information beyond coincidence.")
    else:
        print(f"\n  FINDING: Real correlation not significantly above null.")
        print(f"  Observed co-movement may be a statistical artifact.")

    return {
        "real_mean_corr"  : real_mean,
        "null_mean_corr"  : null_mean,
        "null_std_corr"   : null_std,
        "excess_corr"     : real_mean - null_mean,
        "z_score"         : float(z_score),
        "p_value"         : p_value,
        "n_permutations"  : N_PERMUTATIONS,
        "spatial_significant": bool(p_value < 0.05),
    }


# ── Test 4: Same-time vs lagged correlation ───────────────────────────

### 5.6 Test 4 — Lead-Lag Correlation

Compares same-time correlation vs lagged correlation across all stock pairs.

- **Same-time:** Stock A at t vs Stock B at t → contemporaneous co-movement
- **Lagged:** Stock A at t → Stock B at t+1 → genuine predictive signal

A large gap between the two means cross-stock information is contemporaneous
and cannot be used to forecast the next period.


In [ ]:
def test_lead_lag(panel: pd.DataFrame) -> dict:
    """
    Compare same-time correlation vs lagged correlation.

    If same-time correlation >> lagged → stocks react simultaneously
    (market-wide factor, useful for nowcasting but not forecasting).

    If lagged correlation is significant → stock A today predicts
    stock B tomorrow (genuine predictive cross-stock signal).
    """
    print("\n" + "="*55)
    print("Test 4: Same-Time vs Lagged Correlation")
    print("="*55)

    obs_count   = panel.notna().sum()
    good_stocks = obs_count[obs_count >= panel.shape[0] * 0.7].index
    sub         = panel[good_stocks].dropna()

    if len(sub) < 10:
        print("Not enough complete rows for lead-lag test.")
        return {}

    n           = len(sub.columns)
    upper_idx   = np.triu_indices(n, k=1)

    # Same-time correlation
    corr_same   = sub.corr().values[upper_idx]
    corr_same   = corr_same[~np.isnan(corr_same)]

    # Lagged correlation: stock A at t predicts stock B at t+1
    sub_t       = sub.iloc[:-1].values
    sub_t1      = sub.iloc[1:].values
    lag_corrs   = []
    for i in range(n):
        for j in range(i+1, n):
            mask = ~(np.isnan(sub_t[:, i]) | np.isnan(sub_t1[:, j]))
            if mask.sum() < 10:
                continue
            r, _ = stats.pearsonr(sub_t[mask, i], sub_t1[mask, j])
            lag_corrs.append(r)
    lag_corrs = np.array(lag_corrs)

    print(f"Same-time correlation  : mean={corr_same.mean():.4f}  "
          f"std={corr_same.std():.4f}")
    print(f"Lagged correlation     : mean={lag_corrs.mean():.4f}  "
          f"std={lag_corrs.std():.4f}")
    print(f"Lag decay ratio        : {lag_corrs.mean()/max(corr_same.mean(),EPS):.3f}")
    print(f"  (1.0 = no decay, 0.0 = all contemporaneous)")

    pct_lag_sig = float((np.abs(lag_corrs) > 0.1).mean())
    print(f"Pairs with |lag corr| > 0.1: {pct_lag_sig*100:.1f}%")

    if lag_corrs.mean() > 0.05:
        print(f"\n  FINDING: Meaningful lagged correlation ({lag_corrs.mean():.3f}).")
        print(f"  Stock A's past RV predicts stock B's future RV.")
        print(f"  Cross-stock signals have genuine PREDICTIVE value.")
    else:
        print(f"\n  FINDING: Weak lagged correlation ({lag_corrs.mean():.3f}).")
        print(f"  Cross-stock co-movement is mostly contemporaneous.")
        print(f"  Useful for nowcasting but limited for forecasting.")

    return {
        "same_time_mean_corr": float(corr_same.mean()),
        "lagged_mean_corr"   : float(lag_corrs.mean()),
        "lag_decay_ratio"    : float(lag_corrs.mean() / max(corr_same.mean(), EPS)),
        "pct_lag_significant": pct_lag_sig,
        "predictive_value"   : bool(lag_corrs.mean() > 0.05),
    }


# ── Main ─────────────────────────────────────────────────────────────

### 5.7 Run All Tests and Summary

In [ ]:
print("=" * 55)
print("Stock Interaction Importance Tests")
print("  Independent of any model — pure statistics")
print("=" * 55)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("\nLoading RV panel ...")
panel = load_rv_panel("train")

results = {}

results["correlation"] = test_correlation(panel)
results["granger"]     = test_granger(panel)
results["permutation"] = test_permutation(panel)
results["lead_lag"]    = test_lead_lag(panel)

# ── Summary ───────────────────────────────────────────────────
print("\n" + "=" * 55)
print("SUMMARY")
print("=" * 55)

c = results["correlation"]
g = results["granger"]
p = results["permutation"]
l = results["lead_lag"]

print(f"\n{'Test':<30} {'Finding':<15} {'Key Stat'}")
print("-" * 55)
print(f"{'1. Cross-stock correlation':<30} "
      f"{'STRONG' if c['mean_corr'] > 0.3 else 'WEAK':<15} "
      f"mean_corr={c['mean_corr']:.3f}")
print(f"{'2. Granger causality':<30} "
      f"{'SIGNIFICANT' if g.get('interaction_significant') else 'NOT SIG':<15} "
      f"pct_sig={g.get('pct_significant', 0)*100:.1f}%")
print(f"{'3. Permutation test':<30} "
      f"{'REAL' if p.get('spatial_significant') else 'RANDOM':<15} "
      f"z={p.get('z_score', 0):.2f}")
print(f"{'4. Lagged correlation':<30} "
      f"{'PREDICTIVE' if l.get('predictive_value') else 'CONTEMPOR.':<15} "
      f"lag_corr={l.get('lagged_mean_corr', 0):.3f}")

# Overall verdict
n_positive = sum([
    c["mean_corr"] > 0.3,
    g.get("interaction_significant", False),
    p.get("spatial_significant", False),
    l.get("predictive_value", False),
])

print(f"\nOverall: {n_positive}/4 tests support cross-stock interactions")
if n_positive >= 3:
    print("VERDICT: Strong evidence — cross-stock signals are worth modelling.")
    print("  Your GNN graph and attention layers should help forecasting.")
elif n_positive >= 2:
    print("VERDICT: Moderate evidence — cross-stock signals exist but are weak.")
    print("  GNN may help on high-volatility days but not overall.")
else:
    print("VERDICT: Weak evidence — stocks behave mostly independently.")
    print("  Per-stock models (LGBM) likely sufficient for this dataset.")

# Save results
out_path = OUTPUT_DIR / "stock_interaction_results.json"
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved -> {out_path}")

---
## Summary

| Component | File | Key Output |
|-----------|------|------------|
| Data split | `split.py` | `processed/{train,test}.parquet` |
| Preprocessing | `preprocess.py` | `processed_v2/{train,val,test}_*.parquet` |
| Noise analysis | `noise_floor.py` | `models/stock_noise_profiles.csv` |
| LightGBM | `LightGBM.py` | `models/lgbm_predictions.csv` (QLIKE ≈ 0.031) |
| Interaction tests | `stock_interaction_test.py` | `models/stock_interaction_results.json` |

### Key Findings
- **LightGBM QLIKE = 0.031** — beats HAR-OLS (0.073) by 58%
- **Stock co-movement is contemporaneous** (same-time corr = 0.763, lag corr = 0.006)
- **Cross-stock predictive signals are weak** — per-stock fixed effects dominate. Could be because the target window is small so microstructure noise dominates leading to less stock interaction.
- **Noise floor** — 99.4% of samples are noise-dominated (SNR < 2) at 100s bucket level
